In [34]:
# Setup

%load_ext autoreload
%autoreload 2

import json
import logging
from pathlib import Path
from typing import Any

from dotenv import load_dotenv
from pydantic import BaseModel

from comments import prompts
from comments.llm_generator import LLMGenerator
from comments.llm_labeling import LLMLabeler
from comments.static_labeling import label_unfinished_comments, label_code_snippets_comments
from shared import DifferencesEvaluator, load_dataset, save_dataset, ManualLabeler
from shared.deduplicate import JsonDeduplicator
from shared.llm_connector import Model
from shared.utils import map_labels, dataset_remove_properties, find_by_key

load_dotenv()

logging.basicConfig(level=logging.INFO)
deduplicated_file_path = None

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [35]:
# Deduplication

input_path = "/media/martin/Big data1/datasets/github-projects/hive.json"
deduplicator = JsonDeduplicator()
file_path = Path(input_path)

deduplicated_file_path = deduplicator.deduplicate_dataset_file(file_path)

In [ ]:
# Labeling unfinished comments - LLM

UNFINISHED_COMMENT_ATTR = "unfinished_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_unfinished: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[UNFINISHED_COMMENT_ATTR] = 1 if response.is_unfinished else 0

labeler = LLMLabeler[ResponseModel](init_prompt=prompts.TODO_COMMENTS_INIT_PROMPT, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=UNFINISHED_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llama-labeled-unfinished")


In [36]:
# Labeling unfinished comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_unfinished_comments(deduplicated_file_path)


In [ ]:
# Labeling unfinished comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="unfinished_comment_label", to_compare_keys=["unfinished_comment_llm", "unfinished_comment_static"], no_conflict_key="unfinished_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Labeling commented code comments - LLM

CODE_COMMENT_ATTR = "code_comment_llm"

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

class ResponseModel(BaseModel):
    is_code_comment: bool

def process_response(response: ResponseModel, element: dict[str, Any]):
    element[CODE_COMMENT_ATTR] = 1 if response.is_code_comment else 0


labeler = LLMLabeler[ResponseModel](init_prompt=prompts.CODE_COMMENTS_INIT_PROMPT_2, run_prompt=prompts.RUN_PROMPT_1,
                                    labeled_attribute=CODE_COMMENT_ATTR, model=Model.LLAMA_3_1,
                                    process_response=process_response, response_class=ResponseModel)
await labeler.label_dataset(deduplicated_file_path, "-llm-labeled")

In [37]:
# Labeling commented code comments - static rules

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

label_code_snippets_comments(deduplicated_file_path)

In [ ]:
# Labeling commented code comments - finding differences

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

evaluator = DifferencesEvaluator(new_key_name="code_comment_label", to_compare_keys=["code_comment_llm", "code_comment_static"], no_conflict_key="code_comment_static")
dataset = load_dataset(deduplicated_file_path)
evaluator.evaluate(dataset)
save_dataset(deduplicated_file_path, dataset)

In [ ]:
# Map LLM a static labels to final label

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

mapping = {
    "code_comment_label": "code_comment",
    "unfinished_comment_label": "technical_debt"
}

dataset = load_dataset(deduplicated_file_path)
dataset = map_labels(mapping, "clean", dataset)

save_dataset(deduplicated_file_path, dataset)


In [ ]:
# Remove unused dataset properties

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)
dataset_remove_properties(["unfinished_comment_llm", "unfinished_comment_static", "unfinished_comment_label", "code_comment_llm", "code_comment_static", "code_comment_label"], dataset)

save_dataset(deduplicated_file_path, dataset)

In [38]:
# Manual labeling

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

labeler = ManualLabeler(lambda el: el["unfinished_comment_static"] == 1, "code_comment_static")
labeler.evaluate_labeled(deduplicated_file_path, )

Dataset length: 59255
Element 99
{
  "commentType": "LINE",
  "text": "TODO, we don't support this, but we should, since users may create an empty partition and\nthen load data into it.\n",
  "unfinished_comment_static": 1,
  "code_comment_static": 0
}
-------
Element 153
{
  "commentType": "LINE",
  "text": "TODO: Fix LoadPartitionDoneEvent. Currently, LPDE can only carry a single partition-spec. And that defeats the purpose.\nif(lpde.getStatus())\nsend(lpde.getPartitionName(),lpde.getTable().getParameters().get(HCatConstants.HCAT_MSGBUS_TOPIC_NAME),HCatConstants.HCAT_PARTITION_DONE_EVENT);\n",
  "unfinished_comment_static": 1,
  "code_comment_static": 1
}
-------
Element 155
{
  "commentType": "JAVADOC",
  "text": "Send dropped partition notifications. Subscribers can receive these notifications for a\nparticular table by listening on a topic named \"dbName.tableName\" with message selector\nstring {@value org.apache.hive.hcatalog.common.HCatConstants#HCAT_EVENT} =\n{@value org.apach

In [ ]:
# LLM - generate TODO comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(prompts.GENERATE_VARIATIONS_TODO, deduplicated_file_path, "todo-generated.json")

def filter_fnc(element):
    return "technical_debt" in element["labels"]

await generator.for_each_element(filter_fnc)

In [ ]:
# LLM - generate code comments

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

generator = LLMGenerator(prompts.GENERATE_VARIATIONS_CODE, deduplicated_file_path, "code-comments-generated.json")

def filter_fnc(element):
    return "code_comment" in element["labels"]

await generator.for_each_element(filter_fnc)

In [ ]:
# Merge generated into the dataset

if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

folder_path = "/media/martin/Big data1/datasets/github-projects/"
# generated_files = ["tomcat-deduplicated-0-todo-generated.json", "tomcat-deduplicated-1-todo-generated.json",
#                    "tomcat-deduplicated-2-todo-generated.json", "tomcat-deduplicated-3-todo-generated.json",
#                    "tomcat-deduplicated-4-todo-generated.json"]

generated_files = ["tomcat-deduplicated-0-code-comments-generated.json", "tomcat-deduplicated-1-code-comments-generated.json",
                   "tomcat-deduplicated-2-code-comments-generated.json", "tomcat-deduplicated-3-code-comments-generated.json",
                   "tomcat-deduplicated-4-code-comments-generated.json"]

# generated_files = ["tomcat-deduplicated-1-todo-generated.json"]
dataset = load_dataset(deduplicated_file_path)
print(len(dataset))

def process_file(p: Path):
    with open(p, "r") as f:
        content = json.load(f)

    for element in content:
        original_value = find_by_key(dataset, "text", element["origin"])

        for generated_text in element["generated"]:
            new_value = original_value.copy()
            new_value["text"] = generated_text
            dataset.append(new_value)

for file in generated_files:
    path = Path(folder_path + file)
    process_file(path)

print(len(dataset))
save_dataset(deduplicated_file_path, dataset)


In [23]:
if deduplicated_file_path is None:
    deduplicated_file_path = Path(input("Path to deduplicated file: "))

dataset = load_dataset(deduplicated_file_path)

len(list(filter(lambda el : "technical_debt" in el["labels"], dataset)))

40